# Scientific Computing Project #1

---------------

**Author:** *Salvador Palma* (mtr765) <br>
**Institution:** UCPH - University of Copenhagen

In [1]:
from watermatrices import Amat, Bmat, yvec
import numpy as np

In [2]:
print(f"Matrix A: {Amat.shape}")
print(f"Matrix B: {Bmat.shape}")
print(f"Vector Y: {yvec.shape}")

Matrix A: (7, 7)
Matrix B: (7, 7)
Vector Y: (7,)


In [3]:
dim = Amat.shape[0]

E = np.block([[Amat, Bmat], [Bmat, Amat]])

I = np.eye(dim)
O = np.zeros((dim, dim))

S = np.block([[I, O], [O, -I]])

z = np.concatenate((yvec, -yvec))

print(f"Matrix E: {E.shape}")
#print(f"Matrix E:\n{E}")
print("\n")

print(f"Matrix S: {S.shape}")
#print(f"Matrix S:\n{S}")
print("\n")

print(f"Vector z: {z.shape}")
print(f"Vector z:\n{z}")

Matrix E: (14, 14)


Matrix S: (14, 14)


Vector z: (14,)
Vector z:
[-0.05677315 -0.00902581  0.16002152  0.07001784  0.67801388 -0.10904168
  0.9050518   0.05677315  0.00902581 -0.16002152 -0.07001784 -0.67801388
  0.10904168 -0.9050518 ]


## Week 1

### Exercise a)

#### (1) 

The function `conditionNumber(M)` follows the equation:

$$ ||A||_\infty = max_i \sum\limits_{j=1}^n |a_{ij}|$$

to compute the matrix norm, and then applies the one given in the exercise:

$$ cond_\infty(M) = ||M||_\infty \cdot ||M^{-1}||_\infty $$

In [4]:
def infNorm(M):
    return np.max(np.sum(np.abs(M), axis=1))

def conditionNumber(M):
    M_inv = np.linalg.inv(M)

    maxValOg = infNorm(M)
    maxValInv = infNorm(M_inv)
    
    return maxValOg * maxValInv

#### (2)

Given that the only non-exact input is the right-hand side `z`, with 8 significant digits, its relative error is bounded by $\frac{||\Delta z||_\infty}{||z||_\infty} \le \frac{1}{2}10^{-8}$, and this error associates to the solution `x` via

$$\frac{||\Delta x||_\infty}{||x||_\infty} \le \text{cond}_\infty(E-\omega S)\cdot\frac{||\Delta z||_\infty}{||z||_\infty} \le \text{cond}_\infty(E-\omega S)\cdot \frac{1}{2}10^{-8}$$

This means that the condition number acts as an amplification factor, that is, it measures how much a relative error in the input increases in the output. To calculate the amount of decimal digits we can guarantee in the solution `x`, we can leverage the following formula based on the relative error just calculated:

$$ \text{Guaranteed Digits} = \lfloor-\log_{10} |E_{rel}|\rfloor$$

which leads us to the results depicted below:

| $\omega$ | $\text{cond}_\infty(E- \omega S) $ | upper bound | guaranteed digits in `x` |
| --- | --- | --- | --- |
| 0.800 | 327.82 | 1.64e-06 | **5** |
| 1.146 | 152679.27 | 7.63e-04 | **3** |
| 1.400 | 227.19 | 1.14e-06 | **5** |

In [5]:
ws = [0.800, 1.146, 1.400]

for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    print(f"w={w:.3f}")
    print(f"condition number = {condN:.2f}")
    print(f"upper bound = {condN*0.5e-8:.2e}")
    print(f"guaranteed digits = {np.floor(-np.log10(abs(condN*0.5e-8)))}")
    print("\n")

w=0.800
condition number = 327.82
upper bound = 1.64e-06
guaranteed digits = 5.0


w=1.146
condition number = 152679.27
upper bound = 7.63e-04
guaranteed digits = 3.0


w=1.400
condition number = 227.19
upper bound = 1.14e-06
guaranteed digits = 5.0




### Exercise b)

#### (1)

For each $\omega$, we determine the bound on the relative forward error via:

$$ \frac{||\Delta x||_\infty}{||\hat{x}||_\infty} \le \text{cond}_\infty (E - \omega S) \cdot \frac{||\delta \omega S||_\infty}{||E - \omega S||_\infty}$$

as $\omega$ is given with 3 decimal digits, we know that the perturbation $\delta \omega$ shall be $\frac{1}{2} \cdot 10^{-3}$, the bounds are as follows:

| $\omega$ | bound |
|---|---|
| 0.800 | 0.00522|
| 1.146 | 2.40504 |
| 1.400 | 0.00355 |

In [6]:
for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    upper = infNorm(0.0005 * S)
    lower = infNorm(M)
    upperBound = condN * (upper / lower)
    print(f"w={w:.3f} upper bound = {upperBound:.5f}")


w=0.800 upper bound = 0.00522
w=1.146 upper bound = 2.40504
w=1.400 upper bound = 0.00355


#### (2)

Using the upper bounds for the relative errors that we just calculated, and the following formula:

$$ \text{Guaranteed Digits} = \lfloor-\log_{10} |E_{rel}|\rfloor$$

We can calculate the amount of significant digits that we can guarantee in `x`, leading to the following results

| $\omega$ | guaranteed digits |
|---|---|
| 0.800 | 2|
| 1.146 | 0 |
| 1.400 | 2 |

*Note: $\omega=1.146$ actually leads to the equation above to equal $-1$, but this is the same as saying we can't guarantee **any** decimal digit*

In [7]:
for w in ws:
    M = E - w * S
    condN = conditionNumber(M)
    upper = infNorm(0.0005 * S)
    lower = infNorm(M)
    upperBound = condN * (upper / lower)
    
    guaranteedDigits = np.floor(-np.log10(abs(upperBound)))
    guaranteedDigits = max(0, guaranteedDigits)
    print(f"w={w:.3f} guaranteed digits = {guaranteedDigits:.0f}")

w=0.800 guaranteed digits = 2
w=1.146 guaranteed digits = 0
w=1.400 guaranteed digits = 2


### Exercise c)

The function `lu_factorize(M)` takes in a square matrix `M` and computes its LU factorization, in such a way that M = LU. In the output cell below we can observe that same verification, where `l @ u` yielded the same matrix as `testMatrix`. 

The functions `forward_substitute(L, b)` and `back_substitute(U, y)` leverage the $LU$ matrices to reach the same solution as the original `Mx = b`. The former yields the solution vector `y` in `Ly = b`, whilst the latter yields the solution vector `x` in `Ux = y`. To verify, we can compare against the direct solution from built-in libraries.

In [8]:
def lu_factorize(M):
    assert M.shape[0] == M.shape[1], "Matrix ain't there, be square"

    n = M.shape[0]
    U = M.astype(float).copy()
    L = np.eye(n)
    for i in range(n):
        if U[i, i] == 0:
            raise ZeroDivisionError(f"zero pivot at index {i}: LU without pivoting fails")
        L[i+1:, i]   = U[i+1:, i] / U[i, i]               # every multiplier of column i at once
        U[i+1:, i:] -= np.outer(L[i+1:, i], U[i, i:])      # rank-1 update of the remaining submatrix
        U[i+1:, i]   = 0.0                                 # exact zeros below the diagonal
    return L, U

testMatrix = np.array([[2, 1, 1], [4, 1, 4], [-6, -5, 3]], dtype=float)
testRHS    = np.array([4, 11, 4], dtype=float)

l, u = lu_factorize(testMatrix)

print("---- Lower ----")
print(l)
print("---- Upper ----")
print(u)
print("---- Check: L @ U ----")
print(l @ u)

---- Lower ----
[[ 1.  0.  0.]
 [ 2.  1.  0.]
 [-3.  2.  1.]]
---- Upper ----
[[ 2.  1.  1.]
 [ 0. -1.  2.]
 [ 0.  0.  2.]]
---- Check: L @ U ----
[[ 2.  1.  1.]
 [ 4.  1.  4.]
 [-6. -5.  3.]]


In [9]:
def forward_substitute(L, b):
    n = L.shape[0]
    y = np.zeros(n, dtype=float)
    for k in range(n):                                     # y[k] = (b[k] - dot(L[k,:k], y[:k])) / L[k,k]
        if L[k, k] == 0:
            raise ZeroDivisionError(f"singular L at index {k}")
        y[k] = (b[k] - L[k, :k] @ y[:k]) / L[k, k]
    return y

def back_substitute(U, y):
    n = U.shape[0]
    x = np.zeros(n, dtype=float)
    for k in range(n-1, -1, -1):                           # x[k] = (y[k] - dot(U[k,k+1:], x[k+1:])) / U[k,k]
        if U[k, k] == 0:
            raise ZeroDivisionError(f"singular U at index {k}")
        x[k] = (y[k] - U[k, k+1:] @ x[k+1:]) / U[k, k]
    return x

y = forward_substitute(l, testRHS)
x = back_substitute(u, y)

print(f"y   (solves L y = b) = {y}")
print(f"x   (solves U x = y) = {x}")
print()
print(f"numpy y              = {np.linalg.solve(l, testRHS)}")
print(f"numpy x              = {np.linalg.solve(testMatrix, testRHS)}")
print(f"max |x - numpy x|    = {np.max(np.abs(x - np.linalg.solve(testMatrix, testRHS))):.2e}")

y   (solves L y = b) = [ 4.  3. 10.]
x   (solves U x = y) = [-4.  7.  5.]

numpy y              = [ 4.  3. 10.]
numpy x              = [-4.  7.  5.]
max |x - numpy x|    = 8.88e-16


Some further sanity checks, since the $3\times3$ system above is dense and well-conditioned and
therefore exercises very little of the code:

### Exercise d)

#### (1)

The function `solve_alpha(omega)` computes $\alpha (\omega)$ by the equation $\alpha (\omega) = \text{z}^T \text{x}$. To find $\text{x}$, it LU-factorizes $E - \omega S$, forward, and back substitutes, all with the functions from the previous exercise, so that

$$ (\mathbf{E}-\omega\mathbf{S})\,\mathbf{x} = \mathbf{z}$$


Then, we compute a table of polarizabilities,  with $\delta\omega = \tfrac{1}{2}\cdot 10^{-3}$:

| $\omega$ |$\alpha(\omega-\delta\omega)$ | $\alpha(\omega)$ | $\alpha(\omega+\delta\omega)$ |
| --- | --- | --- | --- |
| 0.800 |1.6278 | 1.6361 |1.6444|
| 1.146 |994.7530 |2609.2353| -4185.0185 |
| 1.400 |-2.7139 | -2.7069 |-2.6999 |

In [95]:
def solve_alpha(omega):
    A = E - omega * S
    L, U = lu_factorize(A)
    y = forward_substitute(L, z)
    x = back_substitute(U, y)

    return z.transpose() @ x

dw = 0.5e-3

for w in ws:
    print(f"w={w:.3f}")
    print(f"a(w-dw) = {solve_alpha(w - dw):.4f}")
    print(f"a(w) = {solve_alpha(w):.4f}")
    print(f"a(w+dw) = {solve_alpha(w + dw):.4f}")
    print("\n")

w=0.800
a(w-dw) = 1.6278
a(w) = 1.6361
a(w+dw) = 1.6444


w=1.146
a(w-dw) = 994.7530
a(w) = 2609.2353
a(w+dw) = -4185.0183


w=1.400
a(w-dw) = -2.7139
a(w) = -2.7069
a(w+dw) = -2.6999




#### (2)

The two bounds at hand answer to different problems. Bound (a) relates to the error in the input `z`, which is given with 8 significant digits, making every computed $\alpha(\omega)$ carry that same constant uncertainty independent of if we are dealing with $\omega$ or its variations $\omega \pm \delta$. Therefore it doesn't answer nothing regarding the variation deu to the perturbation $\delta$. Bound (b), on the other hand, is another story. The perturbation $\delta$ changes the matrix $E - \omega S$ by $- \delta S$, rather than the right-hand side `z`. And the bound given in (b) is exactly what relates this difference to the final output `x`.

To sum up, the error-bound that is correct to understand the variation of the calculated polarizabilities due to the perturbation $\delta$ is (b) and (b) only.

#### (3)

As stated just now, $\omega$ does not affect the input `z`, therefore, we can consider the `x` in $\alpha(\omega) = z^T x$ as a function of $\omega$, that is $x(\omega)$. With this in mind, we can expand the $\Delta \alpha(\omega)$ expression to

$$\Delta \alpha(\omega) = \alpha(\omega + \delta\omega) - \alpha(\omega) = z^Tx(\omega + \delta\omega) - z^Tx(\omega) = z^T \Delta x$$

in which $\Delta x = x(\omega+\delta\omega) - x(\omega)$


The target form has $| \alpha( \omega) |$, which expanding into the sum of the individual entries of `z` and `x` lets us say that:

$$| \alpha( \omega) | = | \sum\limits_i z_i \Delta x_i|  $$

Considering the summatory triangle inequality and the fact that any element of `x` is lesser or equal than the norm (i.e. $|x_i| \le ||x||_\infty$), we develop into

$$ | \sum\limits_i z_i \Delta x_i|  \le \sum\limits_i |z_i| |\Delta x_i| \le (\sum\limits_i |z_i|) ||\Delta x||_\infty$$


In the equation in (b), we saw that the matrix is disturbed by $\delta \omega S$. However, since the matrix `S` is only represented by either 1 or -1, we know that its norm will simply be the absolute of the perturbation, that is, $||\delta \omega S ||_\infty = |\delta\omega| $. So we can rewrite the equation as

$$ \frac{||\Delta x||_\infty}{||\hat{x}||_\infty} \le \text{cond}_\infty (E - \omega S) \cdot \frac{|\delta \omega |}{||E - \omega S||_\infty}$$

And with some more basic math, we get

$$ \equiv ||\Delta x||_\infty \le ||\hat{x}||_\infty \cdot \text{cond}_\infty (E - \omega S) \cdot \frac{|\delta \omega |}{||E - \omega S||_\infty}$$

$$ \equiv (\sum\limits_i |z_i|) \cdot ||\Delta x||_\infty \le (\sum\limits_i |z_i|) \cdot ||\hat{x}||_\infty \cdot \text{cond}_\infty (E - \omega S) \cdot \frac{|\delta \omega |}{||E - \omega S||_\infty}$$

$$ \equiv | \alpha( \omega) | \le (\sum\limits_i |z_i|) \cdot ||\hat{x}||_\infty \cdot \text{cond}_\infty (E - \omega S) \cdot \frac{|\delta \omega |}{||E - \omega S||_\infty}$$

$$ \equiv | \alpha( \omega) | \le B(\omega) \cdot | \delta \omega |, \qquad B(\omega) = (\sum\limits_i |z_i|) \cdot ||\hat{x}||_\infty \cdot   \frac{\text{cond}_\infty (E - \omega S)}{||E - \omega S||_\infty}$$

Which is the exact bound form we aimed for!

After implementing it in code, we get the following values, meaning our calculated values do fall within the bound correctly


| $\omega$ | $\delta\omega$ |  $B(\omega)\,\lvert\delta\omega\rvert$ | $\lvert\Delta\alpha\rvert$ | Passed |
| --- | --- | --- | --- | --- |
| 0.800 | + | 6.59e-02 | 8.29e-03 | Yes | 
| 0.800 |- | 6.59e-02 | 8.32e-03 |Yes |
| 1.146 | + | 2.72e+04 | 6.79e+03 | Yes |
| 1.146 |- | 2.72e+04 | 1.61e+03 | Yes |
| 1.400 | + | 4.58e-02 | 6.97e-03 | Yes|
| 1.400 | - | 4.58e-02 | 6.99e-03 | Yes|

In [115]:
for w in ws:
    for variation in (+1, -1):
        #alpha calc
        a0 = solve_alpha(w)
        a1 = solve_alpha(w + variation * dw)
        da = a1 - a0

        #Bound calc
        z_sum = np.sum(np.abs(z))
        
        M = E - w * S
        L, U = lu_factorize(M)
        x_computed = back_substitute(U, forward_substitute(L, z))

        condN = conditionNumber(M)
        norm = infNorm(M)

        B = z_sum * np.max(np.abs(x_computed)) * condN / norm

        bound = B * abs(variation * dw)
        value = abs(da)

        print(f"Bound: {bound:.2e}")
        print(f"Value: {value:.2e}")

        print(f"w={w:.3f}{'+' if variation == 1 else ''}{variation * dw}: {'Passed' if value <= bound else 'Failed'}")
        print("\n")

    print("\n")

Bound: 6.59e-02
Value: 8.29e-03
w=0.800+0.0005: Passed


Bound: 6.59e-02
Value: 8.32e-03
w=0.800-0.0005: Passed




Bound: 2.72e+04
Value: 6.79e+03
w=1.146+0.0005: Passed


Bound: 2.72e+04
Value: 1.61e+03
w=1.146-0.0005: Passed




Bound: 4.58e-02
Value: 6.97e-03
w=1.400+0.0005: Passed


Bound: 4.58e-02
Value: 6.99e-03
w=1.400-0.0005: Passed




